In [ ]:
import pandas as pd
import re
import torch
from transformers import GPTNeoForCausalLM, GPT2Tokenizer

# Load the dataset
file_path = '/newalternatedataset.csv'  # Update this path if needed
df = pd.read_csv(file_path)

# Text Preprocessing Function
def preprocess_text(text):
    text = text.lower()  # Convert to lowercase
    text = re.sub(r'\s+', ' ', text)  # Remove extra whitespace
    text = re.sub(r'[^\w\s]', '', text)  # Remove punctuation
    return text.strip()

# Apply preprocessing to the text column
df['processed_text'] = df['text'].apply(preprocess_text)

# Load GPT-Neo model and tokenizer
model_name = 'EleutherAI/gpt-neo-125M'  # You can choose a larger model if needed
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPTNeoForCausalLM.from_pretrained(model_name)

# Set the padding token to be the EOS token
tokenizer.pad_token = tokenizer.eos_token  # Use EOS token for padding

# Move model to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
model.eval()  # Set the model to evaluation mode

# Function to generate embeddings for a batch of texts
def generate_embeddings_batch(texts):
    inputs = tokenizer(texts, return_tensors='pt', padding=True, truncation=True, max_length=512).to(device)
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True) # This line is changed to include output_hidden_states=True
        # Use the hidden states or specific token embeddings for the embedding
        # Access the last hidden state from the hidden_states tuple
        embeddings = outputs.hidden_states[-1].mean(dim=1)  # Average pooling # This line is changed
    return embeddings.cpu().numpy().tolist()  # Move back to CPU and convert to list

# Generate embeddings in batches
batch_size = 8  # Adjust based on GPU memory
embeddings = []
for i in range(0, len(df['processed_text']), batch_size):
    batch_texts = df['processed_text'][i:i + batch_size].tolist()
    batch_embeddings = generate_embeddings_batch(batch_texts)
    embeddings.extend(batch_embeddings)

# Add embeddings to the DataFrame
df['embedding'] = embeddings

# Drop the original text and processed_text columns
df = df.drop(columns=['text', 'processed_text'])

# Save to a new CSV file
df.to_csv('/content/embedded_gptneo.csv', index=False)

print("Embedding complete and saved as 'embedded_gptneo.csv'")


Embedding complete and saved as 'embedded_gptneo.csv'


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

# Load your dataset
data = pd.read_csv('embedded_gptneo.csv')


data['embedding'] = data['embedding'].apply(lambda x: np.fromstring(x[1:-1], sep=','))
X = np.array(data['embedding'].tolist())
y = data[['emotionaldistress', 'provokingviolence', 'individualharrassment']].values

# Convert y to binary format (multi-label)
y_binary = (y > 0).astype(int)

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y_binary, test_size=0.2, random_state=42)

# Custom Dataset class
class MultilabelDataset(Dataset):
    def __init__(self, embeddings, labels):
        self.embeddings = embeddings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids': torch.tensor(self.embeddings[idx], dtype=torch.float32),
            'labels': torch.tensor(self.labels[idx], dtype=torch.float32)
        }

# Create DataLoaders for training and validation
train_dataset = MultilabelDataset(X_train, y_train)
val_dataset = MultilabelDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

# Hierarchical CNN-BiLSTM Attention Model
class HierarchicalCNNBiLSTMAttention(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(HierarchicalCNNBiLSTMAttention, self).__init__()

        # CNN layers for feature extraction
        self.conv1 = nn.Conv1d(in_channels=1, out_channels=128, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(128, 256, kernel_size=3, padding=1)
        self.pool = nn.MaxPool1d(2)

        # BiLSTM layer
        self.bilstm = nn.LSTM(input_size=256, hidden_size=128, num_layers=1, batch_first=True, bidirectional=True)

        # Fully connected layer after LSTM
        self.fc_shared = nn.Linear(128 * 2 * (input_dim // 2), 128)  # Adjusted for BiLSTM output

        # Attention mechanism and output layer for each label
        self.attention_emotional = nn.Linear(128, 1)
        self.fc_emotional = nn.Linear(128, 1)

        self.attention_violence = nn.Linear(128, 1)
        self.fc_violence = nn.Linear(128, 1)

        self.attention_harassment = nn.Linear(128, 1)
        self.fc_harassment = nn.Linear(128, 1)

    def forward(self, x):
        # Apply CNN layers
        x = x.unsqueeze(1)  # Add channel dimension for Conv1d
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        x = self.pool(x)  # Shape: [batch_size, 256, input_dim / 2]

        # Apply BiLSTM
        x = x.permute(0, 2, 1)  # Reshape for LSTM: [batch_size, seq_len, features]
        lstm_out, _ = self.bilstm(x)  # Shape: [batch_size, seq_len, 256]

        # Flatten the LSTM output for the fully connected layer
        x = lstm_out.contiguous().view(lstm_out.size(0), -1)
        shared_features = torch.relu(self.fc_shared(x))  # Shape: [batch_size, 128]

        # Emotional Distress Classification with Attention
        attn_weights_emotional = torch.softmax(self.attention_emotional(shared_features), dim=1)
        emotional_features = attn_weights_emotional * shared_features
        emotional_output = torch.sigmoid(self.fc_emotional(emotional_features))

        # Provoking Violence Classification with Attention
        attn_weights_violence = torch.softmax(self.attention_violence(shared_features), dim=1)
        violence_features = attn_weights_violence * shared_features
        violence_output = torch.sigmoid(self.fc_violence(violence_features))

        # Individual Harassment Classification with Attention
        attn_weights_harassment = torch.softmax(self.attention_harassment(shared_features), dim=1)
        harassment_features = attn_weights_harassment * shared_features
        harassment_output = torch.sigmoid(self.fc_harassment(harassment_features))

        # Concatenate the outputs for multi-label classification
        return torch.cat((emotional_output, violence_output, harassment_output), dim=1)

# Instantiate the model, loss function, and optimizer
input_dim = X.shape[1]  # Number of features in embeddings
output_dim = y_binary.shape[1]  # Number of labels

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = HierarchicalCNNBiLSTMAttention(input_dim=input_dim, output_dim=output_dim).to(device)

# Loss function and optimizer
criterion = nn.BCELoss()  # Binary Cross-Entropy Loss for multi-label classification
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001)

# Training function
def train_model(model, train_loader, criterion, optimizer, epochs=10):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for batch in train_loader:
            optimizer.zero_grad()
            input_ids = batch['input_ids'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids)
            loss = criterion(outputs, labels)
            total_loss += loss.item()

            loss.backward()
            optimizer.step()

        print(f"Epoch {epoch + 1}/{epochs}, Loss: {total_loss / len(train_loader):.4f}")

# Train the model
train_model(model, train_loader, criterion, optimizer)

# Evaluation function
def evaluate_model(model, val_loader):
    model.eval()
    predictions, true_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            labels = batch['labels'].cpu().numpy()

            outputs = model(input_ids).cpu().numpy()
            predictions.append(outputs)
            true_labels.append(labels)

    return np.vstack(predictions), np.vstack(true_labels)

# Evaluate the model
y_pred, y_true = evaluate_model(model, val_loader)

# Binarize predictions
y_pred_binary = (y_pred > 0.5).astype(int)

# Print classification report
print(classification_report(y_true, y_pred_binary, target_names=['Emotional Distress', 'Provoking Violence', 'Individual Harassment']))

# Calculate overall accuracy
overall_accuracy = accuracy_score(y_true, y_pred_binary)
print(f"Overall Accuracy: {overall_accuracy:.4f}")


Epoch 1/10, Loss: 0.1672
Epoch 2/10, Loss: 0.1524
Epoch 3/10, Loss: 0.1479
Epoch 4/10, Loss: 0.1424
Epoch 5/10, Loss: 0.1349
Epoch 6/10, Loss: 0.1243
Epoch 7/10, Loss: 0.1137
Epoch 8/10, Loss: 0.1021
Epoch 9/10, Loss: 0.0950
Epoch 10/10, Loss: 0.0881
                       precision    recall  f1-score   support

   Emotional Distress       0.99      1.00      1.00     10887
   Provoking Violence       0.85      0.93      0.88      9012
Individual Harassment       0.99      1.00      1.00     10906

            micro avg       0.95      0.98      0.96     30805
            macro avg       0.94      0.97      0.96     30805
         weighted avg       0.95      0.98      0.96     30805
          samples avg       0.95      0.97      0.95     30805

Overall Accuracy: 0.7979


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [3]:
# Define a function to display examples from the validation set
def display_examples(X, y_true, num_examples=10):
    y_pred = (X > 0.5).astype(int)  # Apply threshold to get binary predictions
    for i in range(num_examples):
        print(f"Example {i + 1}:")
        print(f"True Labels: {y_true[i]}")
        print(f"Predicted Labels: {y_pred[i]}\n")

# Display examples from the validation set
display_examples(y_pred_binary, y_val, num_examples=10)

Example 1:
True Labels: [1 1 1]
Predicted Labels: [1 1 1]

Example 2:
True Labels: [1 1 1]
Predicted Labels: [1 0 1]

Example 3:
True Labels: [1 1 1]
Predicted Labels: [1 1 1]

Example 4:
True Labels: [1 1 1]
Predicted Labels: [1 1 1]

Example 5:
True Labels: [1 1 1]
Predicted Labels: [1 1 1]

Example 6:
True Labels: [1 1 1]
Predicted Labels: [1 0 1]

Example 7:
True Labels: [1 0 1]
Predicted Labels: [1 0 1]

Example 8:
True Labels: [1 1 1]
Predicted Labels: [1 1 1]

Example 9:
True Labels: [1 1 1]
Predicted Labels: [1 1 1]

Example 10:
True Labels: [1 1 1]
Predicted Labels: [1 1 1]



In [ ]:
import pandas as pd
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from transformers import AdamW

# Load the dataset containing GPT-Neo embeddings
data = pd.read_csv('embedded_gptneo.csv')

# Convert the 'embedding' column to numpy arrays
data['embedding'] = data['embedding'].apply(lambda x: np.fromstring(x[1:-1], sep=','))
X = np.array(data['embedding'].tolist())  # Convert to numpy array
y = data[['emotionaldistress', 'provokingviolence', 'individualharrassment']].values

# Convert y to binary format (multi-label)
y_binary = (y > 0).astype(int)

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y_binary, test_size=0.2, random_state=42)

# Create a custom Dataset class
class MultilabelDataset(Dataset):
    def __init__(self, embeddings, labels):
        self.embeddings = embeddings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids': torch.tensor(self.embeddings[idx], dtype=torch.float32),
            'labels': torch.tensor(self.labels[idx], dtype=torch.float32)
        }

# Create DataLoaders
train_dataset = MultilabelDataset(X_train, y_train)
val_dataset = MultilabelDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

# Define a transformer-based model for multilabel classification
class TransformerMultilabelClassifier(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(TransformerMultilabelClassifier, self).__init__()
        self.fc1 = nn.Linear(input_dim, 256)  # First layer
        self.dropout = nn.Dropout(0.3)  # Dropout layer
        self.fc2 = nn.Linear(256, output_dim)  # Second layer

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)  # Apply dropout
        x = torch.sigmoid(self.fc2(x))  # Sigmoid for multilabel output
        return x

# Instantiate the model
input_dim = X.shape[1]  # Number of features from GPT-Neo embeddings
output_dim = y_binary.shape[1]  # Number of labels
model = TransformerMultilabelClassifier(input_dim, output_dim)

# Define loss function and optimizer
criterion = nn.BCELoss()  # Binary Cross Entropy Loss for multilabel
optimizer = AdamW(model.parameters(), lr=0.001)

# Move the model to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

# Training function
def train_model(model, train_loader, criterion, optimizer, epochs=10):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for batch in train_loader:
            optimizer.zero_grad()
            input_ids = batch['input_ids'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids)
            loss = criterion(outputs, labels)
            total_loss += loss.item()

            loss.backward()
            optimizer.step()

        print(f"Epoch {epoch + 1}/{epochs}, Loss: {total_loss / len(train_loader):.4f}")

# Train the model
train_model(model, train_loader, criterion, optimizer)

# Evaluation function
def evaluate_model(model, val_loader):
    model.eval()
    predictions, true_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids)
            predictions.append(outputs.cpu().numpy())
            true_labels.append(labels.cpu().numpy())

    return np.vstack(predictions), np.vstack(true_labels)

# Evaluate the model
y_pred, y_true = evaluate_model(model, val_loader)

# Binarize predictions
y_pred_binary = (y_pred > 0.5).astype(int)

# Print classification report
print(classification_report(y_true, y_pred_binary, target_names=['Emotional Distress', 'Provoking Violence', 'Individual Harassment']))

# Calculate overall accuracy
overall_accuracy = accuracy_score(y_true, y_pred_binary)
print(f"Overall Accuracy: {overall_accuracy:.4f}")

# Define a function to display examples from the validation set
def display_examples(X, y_true, num_examples=10):
    y_pred = (X > 0.5).astype(int)  # Apply threshold to get binary predictions
    for i in range(num_examples):
        print(f"Example {i + 1}:")
        print(f"True Labels: {y_true[i]}")
        print(f"Predicted Labels: {y_pred[i]}\n")

# Display examples from the validation set
display_examples(y_pred_binary, y_val, num_examples=10)


/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Epoch 1/10, Loss: 0.1675
Epoch 2/10, Loss: 0.1610
Epoch 3/10, Loss: 0.1591
Epoch 4/10, Loss: 0.1573
Epoch 5/10, Loss: 0.1556
Epoch 6/10, Loss: 0.1544
Epoch 7/10, Loss: 0.1540
Epoch 8/10, Loss: 0.1547
Epoch 9/10, Loss: 0.1538
Epoch 10/10, Loss: 0.1518
                       precision    recall  f1-score   support

   Emotional Distress       0.99      1.00      1.00     10887
   Provoking Violence       0.83      0.99      0.90      9012
Individual Harassment       0.99      1.00      1.00     10906

            micro avg       0.94      1.00      0.97     30805
            macro avg       0.94      1.00      0.96     30805
         weighted avg       0.94      1.00      0.97     30805
          samples avg       0.94      0.99      0.96     30805

Overall Accuracy: 0.8210
Example 1:
True Labels: [1 1 1]
Predicted Labels: [1 1 1]

Example 2:
True Labels: [1 1 1]
Predicted Labels: [1 1 1]

Example 3:
True Labels: [1 1 1]
Predicted Labels: [1 1 1]

Example 4:
True Labels: [1 1 1]
Predicte

/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
import xgboost as xgb

# Load the dataset
data = pd.read_csv('embedded_gptneo.csv')

# Convert the 'embedded_text' column to numpy arrays
data['embedding'] = data['embedding'].apply(lambda x: np.fromstring(x[1:-1], sep=','))
X = np.array(data['embedding'].tolist())

# Define the target columns and initialize label encoders for each
target_columns = ['provokingviolence', 'individualharrassment', 'emotionaldistress']
label_encoders = {col: LabelEncoder() for col in target_columns}

# Encode the labels for each target column
y_encoded = {}
for col in target_columns:
    y_encoded[col] = label_encoders[col].fit_transform(data[col])

# Split data into training and validation sets for each target column
train_test_splits = {}
for col in target_columns:
    X_train, X_val, y_train, y_val = train_test_split(X, y_encoded[col], test_size=0.2, random_state=42)
    train_test_splits[col] = (X_train, X_val, y_train, y_val)

# Function to train and evaluate XGBoost for each target
def train_evaluate_xgboost(target_column):
    X_train, X_val, y_train, y_val = train_test_splits[target_column]

    # Initialize XGBoost classifier with suitable parameters
    model = xgb.XGBClassifier(
        objective='multi:softmax',
        num_class=len(label_encoders[target_column].classes_),  # Number of classes for the target
        eval_metric='mlogloss',
        use_label_encoder=False,
        max_depth=6,
        learning_rate=0.1,
        n_estimators=100,
        random_state=42
    )

    # Train the model
    model.fit(X_train, y_train)

    # Predict on validation data
    y_pred = model.predict(X_val)

    # Convert predictions and true labels back to original labels
    y_pred_labels = label_encoders[target_column].inverse_transform(y_pred)
    y_val_labels = label_encoders[target_column].inverse_transform(y_val)

    # Print classification report and accuracy
    print(f"Classification Report for '{target_column}':")
    print(classification_report(y_val_labels, y_pred_labels))
    accuracy = accuracy_score(y_val_labels, y_pred_labels)
    print(f"Overall Accuracy for '{target_column}': {accuracy:.4f}\n")

    return model

# Train and evaluate XGBoost model for each target column
models = {}
for col in target_columns:
    print(f"Training and evaluating model for target: {col}")
    models[col] = train_evaluate_xgboost(col)


Training and evaluating model for target: provokingviolence


/usr/local/lib/python3.10/dist-packages/xgboost/core.py:158: UserWarning: [16:41:38] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Classification Report for 'provokingviolence':
              precision    recall  f1-score   support

           0       0.50      0.22      0.31      1975
           1       0.15      0.01      0.01       966
           2       0.62      0.86      0.72      5855
           3       0.75      0.69      0.72      2191

    accuracy                           0.63     10987
   macro avg       0.51      0.44      0.44     10987
weighted avg       0.58      0.63      0.58     10987

Overall Accuracy for 'provokingviolence': 0.6347

Training and evaluating model for target: individualharrassment


/usr/local/lib/python3.10/dist-packages/xgboost/core.py:158: UserWarning: [16:45:36] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Classification Report for 'individualharrassment':
              precision    recall  f1-score   support

           0       0.20      0.01      0.02        81
           1       0.51      0.29      0.37      2386
           2       0.53      0.77      0.63      5430
           3       0.56      0.32      0.40      3090

    accuracy                           0.53     10987
   macro avg       0.45      0.35      0.36     10987
weighted avg       0.53      0.53      0.51     10987

Overall Accuracy for 'individualharrassment': 0.5342

Training and evaluating model for target: emotionaldistress


/usr/local/lib/python3.10/dist-packages/xgboost/core.py:158: UserWarning: [16:48:53] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Classification Report for 'emotionaldistress':
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       100
           1       0.59      0.40      0.48      3151
           2       0.78      0.90      0.84      7736

    accuracy                           0.75     10987
   macro avg       0.46      0.43      0.44     10987
weighted avg       0.72      0.75      0.73     10987

Overall Accuracy for 'emotionaldistress': 0.7477



/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import pandas as pd
import numpy as np

# Check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load dataset
data_file = 'embedded_gptneo.csv'
data = pd.read_csv(data_file)

# Assume your dataset has embeddings and multiclass labels
texts = data['embedding'].tolist()  # Precomputed embeddings as strings
texts = [list(map(float, t.strip('[]').split(','))) for t in texts]  # Convert to list of floats
labels_provoking = data['provokingviolence'].values
labels_harassment = data['individualharrassment'].values
labels_distress = data['emotionaldistress'].values

# Hyperparameters
INPUT_DIM = len(texts[0])  # Dimensionality of your embeddings
HIDDEN_DIM = 128
OUTPUT_DIM_PROVOKING = len(np.unique(labels_provoking))
OUTPUT_DIM_HARASSMENT = len(np.unique(labels_harassment))
OUTPUT_DIM_DISTRESS = len(np.unique(labels_distress))
BATCH_SIZE = 64
EPOCHS = 10
LEARNING_RATE = 1e-3

# Dataset Class
class TextDataset(Dataset):
    def __init__(self, X, y_provoking, y_harassment, y_distress):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y_provoking = torch.tensor(y_provoking, dtype=torch.long)
        self.y_harassment = torch.tensor(y_harassment, dtype=torch.long)
        self.y_distress = torch.tensor(y_distress, dtype=torch.long)

    def __len__(self):
        return len(self.y_provoking)

    def __getitem__(self, idx):
        return self.X[idx], self.y_provoking[idx], self.y_harassment[idx], self.y_distress[idx]

# Prepare data
X_train, X_test, y_train_provoking, y_test_provoking = train_test_split(texts, labels_provoking, test_size=0.3, random_state=42)
_, _, y_train_harassment, y_test_harassment = train_test_split(texts, labels_harassment, test_size=0.3, random_state=42)
_, _, y_train_distress, y_test_distress = train_test_split(texts, labels_distress, test_size=0.3, random_state=42)

train_dataset = TextDataset(X_train, y_train_provoking, y_train_harassment, y_train_distress)
test_dataset = TextDataset(X_test, y_test_provoking, y_test_harassment, y_test_distress)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# BiLSTM Model for Multi-Output
class MultiOutputBiLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim_provoking, output_dim_harassment, output_dim_distress):
        super(MultiOutputBiLSTM, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, bidirectional=True, batch_first=True)
        self.fc_provoking = nn.Linear(hidden_dim * 2, output_dim_provoking)
        self.fc_harassment = nn.Linear(hidden_dim * 2, output_dim_harassment)
        self.fc_distress = nn.Linear(hidden_dim * 2, output_dim_distress)

    def forward(self, x):
        _, (hidden, _) = self.lstm(x.unsqueeze(1))  # Add sequence dimension
        hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)  # Concatenate forward and backward hidden states
        output_provoking = self.fc_provoking(hidden)
        output_harassment = self.fc_harassment(hidden)
        output_distress = self.fc_distress(hidden)
        return output_provoking, output_harassment, output_distress

# Initialize model, loss, and optimizer
model = MultiOutputBiLSTM(INPUT_DIM, HIDDEN_DIM, OUTPUT_DIM_PROVOKING, OUTPUT_DIM_HARASSMENT, OUTPUT_DIM_DISTRESS).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# Training loop
def train_model(model, train_loader, criterion, optimizer):
    model.train()
    for epoch in range(EPOCHS):
        total_loss = 0
        for X_batch, y_provoking, y_harassment, y_distress in train_loader:
            X_batch, y_provoking, y_harassment, y_distress = (
                X_batch.to(device), y_provoking.to(device), y_harassment.to(device), y_distress.to(device)
            )
            optimizer.zero_grad()
            pred_provoking, pred_harassment, pred_distress = model(X_batch)
            loss_provoking = criterion(pred_provoking, y_provoking)
            loss_harassment = criterion(pred_harassment, y_harassment)
            loss_distress = criterion(pred_distress, y_distress)
            total_loss = loss_provoking + loss_harassment + loss_distress
            total_loss.backward()
            optimizer.step()
        print(f"Epoch {epoch + 1}/{EPOCHS}, Loss: {total_loss.item():.4f}")

# Evaluation loop
def evaluate_model(model, test_loader):
    model.eval()
    y_true_provoking, y_pred_provoking = [], []
    y_true_harassment, y_pred_harassment = [], []
    y_true_distress, y_pred_distress = [], []
    with torch.no_grad():
        for X_batch, y_provoking, y_harassment, y_distress in test_loader:
            X_batch, y_provoking, y_harassment, y_distress = (
                X_batch.to(device), y_provoking.to(device), y_harassment.to(device), y_distress.to(device)
            )
            pred_provoking, pred_harassment, pred_distress = model(X_batch)
            y_true_provoking.extend(y_provoking.cpu().numpy())
            y_pred_provoking.extend(torch.argmax(pred_provoking, dim=1).cpu().numpy())
            y_true_harassment.extend(y_harassment.cpu().numpy())
            y_pred_harassment.extend(torch.argmax(pred_harassment, dim=1).cpu().numpy())
            y_true_distress.extend(y_distress.cpu().numpy())
            y_pred_distress.extend(torch.argmax(pred_distress, dim=1).cpu().numpy())
    return (y_true_provoking, y_pred_provoking,
            y_true_harassment, y_pred_harassment,
            y_true_distress, y_pred_distress)

# Train the model
train_model(model, train_loader, criterion, optimizer)

# Evaluate the model
(y_true_provoking, y_pred_provoking,
 y_true_harassment, y_pred_harassment,
 y_true_distress, y_pred_distress) = evaluate_model(model, test_loader)

# Print classification reports
print("Classification Report for Provoking Violence:")
print(classification_report(y_true_provoking, y_pred_provoking))

print("Classification Report for Individual Harassment:")
print(classification_report(y_true_harassment, y_pred_harassment))

print("Classification Report for Emotional Distress:")
print(classification_report(y_true_distress, y_pred_distress))


Using device: cuda
Epoch 1/10, Loss: 3.0571
Epoch 2/10, Loss: 2.0224
Epoch 3/10, Loss: 2.0905
Epoch 4/10, Loss: 2.2423
Epoch 5/10, Loss: 2.2738
Epoch 6/10, Loss: 2.7110
Epoch 7/10, Loss: 2.6493
Epoch 8/10, Loss: 2.0100
Epoch 9/10, Loss: 2.2406
Epoch 10/10, Loss: 2.3994
Classification Report for Provoking Violence:
              precision    recall  f1-score   support

           0       0.45      0.32      0.37      2937
           1       0.14      0.00      0.00      1439
           2       0.63      0.83      0.72      8790
           3       0.77      0.65      0.70      3314

    accuracy                           0.63     16480
   macro avg       0.50      0.45      0.45     16480
weighted avg       0.58      0.63      0.59     16480

Classification Report for Individual Harassment:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       125
           1       0.49      0.40      0.44      3589
           2       0.54      0.74    

/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/m